In [5]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv(r"/content/employees.csv")

In [3]:
cols = [
    "Work-Life Balance",
    "Job Satisfaction",
    "Performance Rating",
    "Education Level",
    "Job Level",
    "Company Size",
    "Company Reputation",
    "Employee Recognition"
]

for col in cols:
    print(f"\n{col}")
    print(sorted(df[col].unique()))


Work-Life Balance
['Excellent', 'Fair', 'Good', 'Poor']

Job Satisfaction
['High', 'Low', 'Medium', 'Very High']

Performance Rating
['Average', 'Below Average', 'High', 'Low']

Education Level
['Associate Degree', 'Bachelor’s Degree', 'High School', 'Master’s Degree', 'PhD']

Job Level
['Entry', 'Mid', 'Senior']

Company Size
['Large', 'Medium', 'Small']

Company Reputation
['Excellent', 'Fair', 'Good', 'Poor']

Employee Recognition
['High', 'Low', 'Medium', 'Very High']


In [4]:
import pandas as pd

# Work-Life Balance
df["Work-Life Balance"] = pd.Categorical(
    df["Work-Life Balance"],
    categories=["Poor", "Fair", "Good", "Excellent"],
    ordered=True
)

# Job Satisfaction
df["Job Satisfaction"] = pd.Categorical(
    df["Job Satisfaction"],
    categories=["Low", "Medium", "High", "Very High"],
    ordered=True
)

# Performance Rating
df["Performance Rating"] = pd.Categorical(
    df["Performance Rating"],
    categories=["Low", "Below Average", "Average", "High"],
    ordered=True
)

# Education Level
df["Education Level"] = pd.Categorical(
    df["Education Level"],
    categories=[
        "High School",
        "Associate Degree",
        "Bachelor’s Degree",
        "Master’s Degree",
        "PhD"
    ],
    ordered=True
)

# Job Level
df["Job Level"] = pd.Categorical(
    df["Job Level"],
    categories=[
        "Entry",
        "Mid",
        "Senior"
    ],
    ordered=True
)

# Company Size
df["Company Size"] = pd.Categorical(
    df["Company Size"],
    categories=[
        "Small",
        "Medium",
        "Large"
    ],
    ordered=True
)

# Company Reputation
df["Company Reputation"] = pd.Categorical(
    df["Company Reputation"],
    categories=[
        "Poor",
        "Fair",
        "Good",
        "Excellent"
    ],
    ordered=True
)

# Employee Recognition
df["Employee Recognition"] = pd.Categorical(
    df["Employee Recognition"],
    categories=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ],
    ordered=True
)

### Buesiness questions

###### What share of employees left overall, and which job role is losing the most people? Don't stop at the number — tell leadership where to look first and why that role is a concern.

In [7]:
jobrole_attrition = (
    pd.crosstab(
        df["Job Role"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

jobrole_attrition.head()

Attrition,Job Role,Left,Stayed
0,Education,48.767403,51.232597
1,Finance,46.927642,53.072358
2,Healthcare,47.510835,52.489165
3,Media,46.848950,53.151050
4,Technology,47.091398,52.908602


In [59]:
fig = px.bar(
    jobrole_attrition.sort_values("Left"),
    x="Left",
    y="Job Role",
    orientation="h",
    text_auto=".1f",
    title="Attrition Rate by Job Role"
)

fig.update_layout(
    xaxis_title="Attrition Rate (%)",
    yaxis_title=""
)

fig.show()

##### inisght:the gap between the highest and lowest role is only 2 percentage points, indicating that attrition is distributed fairly evenly across departments rather than being concentrated in one specific function.

#### CTA: 1. Do not launch role-specific retention programs.

The data does not support targeting one department over another.

######  BUSINESS QUESTION :
Are employees who work overtime more likely to leave, and by how much versus those who don't? What should HR do about workload?

In [10]:
overtime_df = (
    pd.crosstab(
        df["Overtime"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

overtime_df

Attrition,Overtime,Left,Stayed
0,No,45.529039,54.470961
1,Yes,51.493365,48.506635


In [11]:
px.bar(
    overtime_df,
    x="Overtime",
    y="Left",
    color="Overtime",
    text_auto=".1f"
)

#### INSIGHT: Insight

Employees who work overtime have an attrition rate of 51.5%, compared to 45.5% for employees who do not work overtime.

This represents a 6 percentage point increase, suggesting that overtime is associated with a higher likelihood of employees leaving the company.

#### CTA: CTA
Monitor employees consistently exceeding a predefined overtime threshold (e.g., 10+ hours/week for 3 consecutive months).
Review teams with the highest overtime levels to identify workload imbalances or staffing shortages.
Introduce workload redistribution or flexible work arrangements for overtime-heavy teams before turnover occurs.
Presentation Version

##### BUSINESS QUESTIONS :Does offering remote work appear to keep people? State the size of the effect — and be honest that only a small share of staff work remotely, so what can and can't be concluded.

In [62]:
remote_population = (
    df["Remote Work"]
    .value_counts()
    .reset_index()
)

remote_population.columns = [
    "Remote Work",
    "Employees"
]

print(remote_population)

  Remote Work  Employees
0          No      60300
1         Yes      14198


In [64]:
remote_attrition = (
    pd.crosstab(
        df["Remote Work"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

fig = px.bar(
    remote_attrition,
    x="Remote Work",
    y="Left",
    color="Remote Work",
    text_auto=".1f",
    title="Attrition Rate by Remote Work Status"
)

fig.update_layout(
    yaxis_title="Attrition Rate (%)",
    xaxis_title=""
)

fig.show()

#### INSIGHT: This is a 28.1 percentage point reduction, making Remote Work one of the strongest retention drivers in the entire analysis. Employees working remotely are approximately half as likely to leave as on-site employees.

#### CTA: CTA
Identify roles that can realistically operate in a hybrid model and prioritize them for remote-work expansion.
Launch a pilot hybrid policy (e.g., 3 office days / 2 remote days) in high-attrition departments and monitor retention over the next quarter.
For roles that must remain on-site, introduce alternative flexibility measures such as flexible hours, compressed workweeks, or shift-swapping options.

##### BUSINESS QUESTION : Within the same job level, do lower-paid employees leave more often? At what point does higher pay stop reducing attrition? Turn this into a recommendation about pay bands.

In [16]:
df["Income Quartile"] = pd.qcut(
    df["Monthly Income"],
    q=4,
    labels=[
        "Low Pay",
        "Lower-Mid Pay",
        "Upper-Mid Pay",
        "High Pay"
    ]
)

In [18]:
pay_fairness = (
    pd.crosstab(
        [df["Job Level"], df["Income Quartile"]],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

pay_fairness.head()

Attrition,Job Level,Income Quartile,Left,Stayed
0,Entry,Low Pay,64.430259,35.569741
1,Entry,Lower-Mid Pay,63.387097,36.612903
2,Entry,Upper-Mid Pay,62.890729,37.109271
3,Entry,High Pay,62.408223,37.591777
4,Mid,Low Pay,46.491812,53.508188


In [19]:
px.line(
    pay_fairness,
    x="Income Quartile",
    y="Left",
    color="Job Level",
    markers=True
)

#### INSIGHT: Insight

Within each Job Level, increasing pay has very little impact on attrition.

Entry-level employees remain around 62–65% attrition across all pay bands.
Mid-level employees remain around 45–46% attrition.
Senior employees remain around 19–21% attrition.

This indicates that Job Level has a much stronger effect on attrition than salary. Simply paying people more within the same level does not significantly reduce turnover.

#### ANSWER : Answer to Q4

Higher pay does not consistently reduce attrition within the same Job Level.

The attrition curves are almost flat, suggesting that after a certain threshold, salary stops being a meaningful retention lever and other factors become more important.

#### CTA: CTA
Do not rely solely on salary increases as a retention strategy for Entry-level employees.
Invest in career progression pathways, mentorship, and promotion opportunities since Job Level appears to drive retention more than pay.
Review pay bands for fairness and competitiveness, but allocate retention budget toward growth and development initiatives rather than across-the-board salary increases.

#####
Q5 · The retention timeline
At what stage of an employee's time at the company is attrition highest? What does that say about where retention effort (onboarding, mid-career, long-tenure) should be aimed?

In [21]:
df["Tenure Bucket"] = pd.cut(
    df["Years at Company"],
    bins=[0, 1, 3, 5, 10, 20, 50],
    labels=[
        "<1 Year",
        "1-3 Years",
        "3-5 Years",
        "5-10 Years",
        "10-20 Years",
        "20+ Years"
    ],
    include_lowest=True
)

In [23]:
timeline = (
    pd.crosstab(
        df["Tenure Bucket"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

timeline

Attrition,Tenure Bucket,Left,Stayed
0,<1 Year,52.225131,47.774869
1,1-3 Years,52.666667,47.333333
2,3-5 Years,53.783197,46.216803
3,5-10 Years,51.622677,48.377323
4,10-20 Years,44.711043,55.288957
5,20+ Years,43.749719,56.250281


In [24]:
px.line(
    timeline,
    x="Tenure Bucket",
    y="Left",
    markers=True
)

##### ANSWER :Answer to Q5

Attrition is highest during the 3–5 year tenure period, reaching approximately 53.8%. After employees pass the 5–10 year mark, attrition begins to decline significantly, dropping to around 44% for employees with 10+ years of service.

#### Insight

The biggest retention challenge is not onboarding and not long-tenure employees — it's employees in their mid-career stage (3–5 years).

This suggests that employees who stay beyond the initial onboarding phase may start questioning their growth opportunities, career progression, or future within the company around the 3–5 year mark.

#### CTA
Launch a Mid-Career Retention Program targeting employees with 3–5 years of tenure.
Require career development discussions and promotion reviews before employees reach their third year.
Offer leadership tracks, skill development plans, and internal mobility opportunities to employees approaching the 3–5 year milestone.

##### Q6 · Engagement warning signs
Combine Job Satisfaction and Work-Life Balance. Which combination of the two is the strongest early-warning sign that someone will leave? Recommend what a manager should watch for.

In [25]:
df["Engagement Profile"] = (
    df["Job Satisfaction"].astype(str)
    + " + "
    + df["Work-Life Balance"].astype(str)
)

engagement_profile = (
    pd.crosstab(
        df["Engagement Profile"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

engagement_profile = engagement_profile.sort_values(
    "Left",
    ascending=False
)

In [26]:
fig = px.bar(
    engagement_profile.head(10),
    x="Left",
    y="Engagement Profile",
    orientation="h",
    text_auto=".1f",
    title="Highest-Risk Satisfaction & Work-Life Balance Profiles"
)

fig.update_layout(
    xaxis_title="Attrition Rate (%)",
    yaxis_title=""
)

fig.show()

##### ANSWER: The strongest early-warning sign of attrition is employees with Low Job Satisfaction + Poor Work-Life Balance, where attrition reaches 67.0%.

Another critical finding is that even employees with Very High Satisfaction + Poor Work-Life Balance still show 64.9% attrition, suggesting that poor work-life balance can outweigh satisfaction.

##### Insight

Work-Life Balance appears to be the dominant factor. Employees may enjoy their work and report high satisfaction, but if they consistently experience poor work-life balance, they are still highly likely to leave.

This means satisfaction surveys alone are not enough to identify retention risk.

#####CTA
Flag employees reporting Poor Work-Life Balance, regardless of their satisfaction score.
Require managers to conduct follow-up discussions with employees who report Poor or Fair balance for two consecutive survey periods.
Monitor workload, overtime, and flexibility options before attrition occurs rather than relying solely on satisfaction metrics.

##### Q7 · Life stage
Do age, marital status, and number of dependents change who leaves? Identify the life-stage group most at risk and propose what would actually retain them.

In [27]:
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[18,25,35,45,60],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-60"
    ]
)

In [28]:
df["Dependent Group"] = pd.cut(
    df["Number of Dependents"],
    bins=[-1,0,2,10],
    labels=[
        "None",
        "1-2",
        "3+"
    ]
)

In [29]:
df["Life Stage"] = (
    df["Age Group"].astype(str)
    + " | "
    + df["Marital Status"]
    + " | "
    + df["Dependent Group"].astype(str)
)

In [38]:
life_stage = (
    pd.crosstab(
        df["Life Stage"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

life_stage = life_stage.sort_values(
    by="Left",
    ascending=False
)

life_stage.head(10)

Attrition,Life Stage,Left,Stayed
44,nan | Single | None,80.681818,19.318182
6,18-25 | Single | 1-2,74.598624,25.401376
42,nan | Single | 1-2,74.261603,25.738397
8,18-25 | Single | None,72.470588,27.529412
43,nan | Single | 3+,71.250000,28.750000
17,26-35 | Single | None,70.368393,29.631607
15,26-35 | Single | 1-2,68.614551,31.385449
35,46-60 | Single | None,67.681499,32.318501
24,36-45 | Single | 1-2,67.252396,32.747604
33,46-60 | Single | 1-2,67.146814,32.853186


In [39]:
fig = px.bar(
    life_stage.head(10),
    x="Left",
    y="Life Stage",
    orientation="h",
    text_auto=".1f",
    title="Top 10 Highest-Risk Life-Stage Profiles"
)

fig.update_layout(
    xaxis_title="Attrition Rate (%)",
    yaxis_title=""
)

fig.show()

#### answer :Yes. Age, marital status, and number of dependents clearly influence attrition.

The highest-risk life-stage group is young single employees (18–35 years old) with no dependents or only 1–2 dependents, where attrition reaches 70–75%.

#### Insight

The data suggests that employees with fewer family responsibilities and fewer personal commitments are much more likely to leave. They have greater flexibility to switch jobs, relocate, or pursue better opportunities.

In contrast, older employees and those with greater family responsibilities tend to stay longer and show lower attrition rates.



##### CTA
Create a dedicated retention program for employees aged 18–35, especially those who are single.
Offer accelerated career development plans, certifications, mentorship, and clear promotion pathways within the first 2–3 years.
Conduct stay interviews with high-performing young employees every 6 months to understand what opportunities they are seeking before they begin looking externally.

##### Q8 · Career stagnation
Build the case that lack of growth drives attrition. Pull together number of promotions, job level, and leadership / innovation opportunities. Does feeling "stuck" line up with leaving? End with a growth / mobility recommendation.

In [41]:
df["Job Level Score"] = (
    df["Job Level"]
    .map({
        "Entry": 1,
        "Mid": 2,
        "Senior": 3
    })
    .astype(int)
)

df["Leadership Score"] = (
    df["Leadership Opportunities"]
    .map({
        "No": 0,
        "Yes": 1
    })
    .astype(int)
)

df["Innovation Score"] = (
    df["Innovation Opportunities"]
    .map({
        "No": 0,
        "Yes": 1
    })
    .astype(int)
)

In [43]:
df["Growth Score"] = (
    df["Number of Promotions"]
    + df["Job Level Score"]
    + df["Leadership Score"]
    + df["Innovation Score"]
)

In [44]:
growth_attrition = (
    pd.crosstab(
        df["Growth Score"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

In [45]:
fig = px.line(
    growth_attrition,
    x="Growth Score",
    y="Left",
    markers=True,
    title="Attrition Rate by Career Growth Score"
)

fig.update_layout(
    xaxis_title="Career Growth Score",
    yaxis_title="Attrition Rate (%)"
)

fig.show()

##### insight:
Employees who receive promotions, move into higher job levels, and gain leadership opportunities are significantly more likely to stay.

The trend is almost perfectly linear: as growth opportunities increase, attrition consistently decreases. This suggests that employees are not just leaving because of pay or workload — many are leaving because they feel their career has stopped progressing.

In other words:

Feeling stuck strongly aligns with leaving.

#### CTA
Establish a formal internal mobility program where employees can apply for new roles, projects, or leadership opportunities after 12–18 months.
Require managers to create documented career development plans for all Entry and Mid-level employees.
Set a KPI that employees should have either a promotion, major skill certification, leadership assignment, or cross-functional project opportunity every 18–24 months.

#### Q9 · The highest-risk profile
Combine 3–4 factors to construct the single highest-risk employee profile you can find. Report (a) how much higher their attrition is than the company average, and (b) how many employees actually match that profile — so leadership knows whether it's worth acting on.

In [47]:
risk_profile = (
    df.groupby([
        "Job Level",
        "Remote Work",
        "Work-Life Balance"
    ])
    .agg(
        Employees=("Employee ID", "count"),
        Attrition_Rate=(
            "Attrition",
            lambda x: (x == "Left").mean() * 100
        )
    )
    .reset_index()
)

/tmp/ipykernel_34233/754950226.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [48]:
risk_profile["Profile"] = (
    risk_profile["Job Level"].astype(str)
    + " | "
    + risk_profile["Remote Work"]
    + " | "
    + risk_profile["Work-Life Balance"].astype(str)
)

In [49]:
risk_profile = risk_profile[
    risk_profile["Employees"] >= 100
]

In [50]:
top_profiles = risk_profile.sort_values(
    "Attrition_Rate",
    ascending=False
).head(10)

fig = px.bar(
    top_profiles,
    x="Attrition_Rate",
    y="Profile",
    orientation="h",
    text="Employees",
    title="Highest-Risk Employee Profiles"
)

fig.update_traces(
    texttemplate="%{text:,} employees",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Attrition Rate (%)",
    yaxis_title=""
)

fig.show()

##### insight:
This is not a small niche group. More than 3,300 employees fall into this profile, and more than 8 out of every 10 employees in this segment leave the company.

The pattern combines the three strongest drivers identified throughout the analysis:

Entry-level employees have the highest attrition.
On-site employees leave far more frequently than remote workers.
Poor work-life balance is one of the strongest predictors of turnover.

When these factors occur together, attrition reaches its highest level in the entire dataset.

##### CTA :
Create a high-risk retention watchlist containing all Entry-level employees who are on-site and report Poor/Fair work-life balance.
Within 30 days, provide this group with at least one flexibility intervention:
Hybrid work eligibility
Flexible scheduling
Reduced overtime targets
Assign a mentor and conduct monthly manager check-ins for all employees in this segment during their first 12 months.
Track attrition in this group quarterly as a dedicated HR KPI.

####
Q10 · What moves the needle
If HR could fix only one thing next quarter to reduce attrition, what does the data say it should be? Rank the top 3 drivers by how much each shifts attrition above the baseline, then defend your #1 pick with a concrete recommendation and a rough estimate of impact.

In [52]:
joblevel_attrition = (
    pd.crosstab(
        df["Job Level"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

remote_attrition = (
    pd.crosstab(
        df["Remote Work"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

wlb_attrition = (
    pd.crosstab(
        df["Work-Life Balance"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

overtime_attrition = (
    pd.crosstab(
        df["Overtime"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

reputation_attrition = (
    pd.crosstab(
        df["Company Reputation"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

recognition_attrition = (
    pd.crosstab(
        df["Employee Recognition"],
        df["Attrition"],
        normalize="index"
    ) * 100
).reset_index()

In [55]:
baseline = (df["Attrition"] == "Left").mean() * 100

drivers = {
    "Job Level": joblevel_attrition,
    "Remote Work": remote_attrition,
    "Work-Life Balance": wlb_attrition,
    "Overtime": overtime_attrition,
    "Company Reputation": reputation_attrition,
    "Employee Recognition": recognition_attrition
}

ranking = []

for driver, table in drivers.items():

    max_shift = abs(table["Left"] - baseline).max()

    ranking.append({
        "Driver": driver,
        "Shift": max_shift
    })

ranking = pd.DataFrame(ranking)

ranking.sort_values(
    "Shift",
    ascending=False,
    inplace=True
)

ranking

,Driver,Shift
0,Job Level,27.211827
1,Remote Work,22.763036
2,Work-Life Balance,12.701423
4,Company Reputation,8.542327
3,Overtime,4.015580
5,Employee Recognition,1.468523


In [57]:
ranking = []

for driver, table in drivers.items():

    max_shift = abs(table["Left"] - baseline).max()

    ranking.append({
        "Driver": driver,
        "Shift": max_shift
    })

ranking = pd.DataFrame(ranking)

ranking.sort_values(
    "Shift",
    ascending=False,
    inplace=True
)

ranking

,Driver,Shift
0,Job Level,27.211827
1,Remote Work,22.763036
2,Work-Life Balance,12.701423
4,Company Reputation,8.542327
3,Overtime,4.015580
5,Employee Recognition,1.468523


In [58]:
fig = px.bar(
    ranking.head(10),
    x="Shift",
    y="Driver",
    orientation="h",
    text_auto=".1f",
    title="Drivers Ranked by Impact on Attrition"
)

fig.update_layout(
    xaxis_title="Maximum Shift from Baseline Attrition (%)",
    yaxis_title=""
)

fig.show()

#### ACTIONS NEED TO BE TAKEN
Answer to Q10

If HR could fix only one thing next quarter, the data suggests it should focus on reducing Entry-Level attrition through stronger career progression and retention programs.

Top 3 Drivers Ranked by Impact
Rank	Driver	Impact on Attrition
🥇 1	Job Level	27.2 percentage points
🥈 2	Remote Work	22.8 percentage points
🥉 3	Work-Life Balance	12.7 percentage points
Why Job Level is #1

The gap between Entry-level and Senior employees is enormous:

Entry-Level Attrition = 63.3%
Senior Attrition = 20.3%
Difference = 43 percentage points

No other factor affects a larger portion of the workforce while also showing such a large attrition gap.

The data consistently showed that Entry-level employees:

Appear in the highest-risk profiles.
Are overrepresented among employees who leave.
Benefit the most from promotions and career growth opportunities.

This aligns with the Career Growth analysis, where attrition dropped from 66% to near 0% as growth opportunities increased.

Concrete Recommendation
Entry-Level Retention Program
Implement a structured 90-day onboarding program.
Assign every Entry-level employee a mentor during their first year.
Conduct mandatory career-path discussions at months 6 and 12.
Publish a clear promotion roadmap showing how employees can progress to Mid-level within 18–24 months.
Rough Impact Estimate

Entry-level employees currently leave at 63.3%.

If HR reduces Entry-level attrition by just 10 percentage points (from 63.3% to 53.3%), thousands of additional employees could be retained annually given the size of the Entry-level population.

Because Entry-level employees represent the largest concentration of turnover, even modest improvements would likely produce a larger company-wide impact than initiatives focused solely on overtime, recognition, or reputation.